# ChakraModel v7 Verification — v3 (finds WHICH FILE is actually imported)

Previous run: file-content check passed (v7 fix confirmed present on disk), but the live `discover()` call still returned raw uncapped video counts. That combination only makes sense if Python is importing a *different* `dataset_discovery.py` than the one just staged. This version prints `dataset_discovery.__file__` and the full `sys.path` right after import, so we see exactly which file is loaded instead of guessing.

1. **Run \u2192 Factory reset session** first.
2. Attach the code dataset, weights dataset, polyp datasets.
3. **Accelerator: None / CPU.**
4. Run All. Paste the full output back.

In [1]:
import os, shutil, glob

WORK = '/kaggle/working'

code_zip = None
for p in glob.glob('/kaggle/input/**/ChakraModel_EvalHarness_Code*.zip', recursive=True):
    code_zip = p
    break

if code_zip:
    print('Unpacking code from', code_zip)
    shutil.unpack_archive(code_zip, WORK)
else:
    eh_dir = None
    for root, dirs, files in os.walk('/kaggle/input'):
        if os.path.basename(root) == 'eval_harness' and 'run_eval.py' in files:
            eh_dir = root
            break
    assert eh_dir, (
        'Could not find ChakraModel_EvalHarness_Code*.zip or an extracted eval_harness/ '
        'folder anywhere under /kaggle/input — attach the code dataset.'
    )
    dataset_root = os.path.dirname(eh_dir)
    print('Found already-extracted code at', dataset_root)
    shutil.copytree(eh_dir, f'{WORK}/eval_harness', dirs_exist_ok=True)
    src_dir = os.path.join(dataset_root, 'src')
    if os.path.isdir(src_dir):
        shutil.copytree(src_dir, f'{WORK}/src', dirs_exist_ok=True)
    print('Copied eval_harness/ and src/ into', WORK)

os.makedirs(f'{WORK}/weights/checkpoints', exist_ok=True)
wanted = {'chakra_transformer_best.pth', 'combo1_best.pth'}
found = set()
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f in wanted and f not in found:
            src = os.path.join(root, f)
            dst = f'{WORK}/weights/checkpoints/{f}'
            shutil.copy(src, dst)
            found.add(f)
            print('Linked', f, 'from', src)
missing = wanted - found
if missing:
    print('WARNING missing weights:', missing)
else:
    print('Both checkpoints staged OK.')

Found already-extracted code at /kaggle/input/datasets/jayasrikannuchamy/chakramodel-evalharness-code-new
Copied eval_harness/ and src/ into /kaggle/working
Linked combo1_best.pth from /kaggle/input/datasets/jayasrikannuchamy/chakratransformer-weights/weights/checkpoints/combo1_best.pth
Linked chakra_transformer_best.pth from /kaggle/input/datasets/jayasrikannuchamy/chakratransformer-weights/weights/checkpoints/chakra_transformer_best.pth
Both checkpoints staged OK.


In [2]:
!pip install -q opencv-python-headless

In [3]:
# ============================================================
# FIND EVERY dataset_discovery.py ANYWHERE ON THE FILESYSTEM
# that Python could possibly resolve, before we even import.
# ============================================================
import subprocess
print("All files named dataset_discovery.py anywhere under / :")
out = subprocess.run(['find', '/', '-name', 'dataset_discovery.py', '-not', '-path', '*/proc/*'],
                      capture_output=True, text=True, timeout=60)
print(out.stdout or '(none found)')
if out.stderr.strip():
    print('stderr (permission noise, usually harmless):')
    print(out.stderr[:2000])

All files named dataset_discovery.py anywhere under / :
/kaggle/input/datasets/jayasrikannuchamy/chakramodel-evalharness-code-new/eval_harness/dataset_discovery.py
/kaggle/working/eval_harness/dataset_discovery.py



In [4]:
# ============================================================
# CPU-ONLY VERIFICATION — no model loading, no GPU needed.
# ============================================================
import sys, os

WORK = '/kaggle/working'
sys.path.insert(0, f'{WORK}/eval_harness')
sys.path.insert(0, f'{WORK}/src')

print("="*70)
print("1. FILE VERSION CHECK (raw file read from disk, exact path)")
print("="*70)

def check(path, must_contain, label):
    if not os.path.exists(path):
        print(f"  [MISSING] {path}")
        return
    src = open(path).read()
    ok = all(s in src for s in must_contain)
    print(f"  [{'OK' if ok else 'FAIL'}] {label}: {os.path.realpath(path)}")
    for s in must_contain:
        print(f"       {'found' if s in src else 'MISSING'}: {s!r}")

check(f'{WORK}/eval_harness/resource_monitor.py',
      ['MAX_BATCH         = 8'], 'batch ceiling fix (v4+)')
check(f'{WORK}/eval_harness/eval_engine.py',
      ['mode="default"', '_key_renames', 'ppd_pred.weight'], 'compile-mode + checkpoint-remap fix (v4/v6+)')
check(f'{WORK}/eval_harness/dataset_discovery.py',
      ['_discover_recursive', 'MAX_FRAMES_PER_VIDEO', 'IterableDataset',
       'total_frames += min(n, MAX_FRAMES_PER_VIDEO)'],
      'recursive discovery + video cap/sequential-decode fix (v7)')

print()
print("="*70)
print("2. sys.path (import search order)")
print("="*70)
for i, p in enumerate(sys.path):
    print(f"  [{i}] {p}")

print()
print("="*70)
print("3. WHICH FILE DOES `import dataset_discovery` ACTUALLY LOAD?")
print("="*70)
for _mod in list(sys.modules):
    if _mod == 'dataset_discovery':
        print(f"  [NOTE] Evicting cached module '{_mod}' from sys.modules before import.")
        del sys.modules[_mod]
import dataset_discovery
print(f"  LOADED FROM: {dataset_discovery.__file__}")
print(f"  Has MAX_FRAMES_PER_VIDEO attr: {hasattr(dataset_discovery, 'MAX_FRAMES_PER_VIDEO')}")
if hasattr(dataset_discovery, 'MAX_FRAMES_PER_VIDEO'):
    print(f"  MAX_FRAMES_PER_VIDEO value: {dataset_discovery.MAX_FRAMES_PER_VIDEO}")
print(f"  Has _discover_recursive attr: {hasattr(dataset_discovery, '_discover_recursive')}")

print()
print("="*70)
print("4. ACTUAL discover() BEHAVIOR")
print("="*70)
metas = dataset_discovery.discover('/kaggle/input')
total = 0
for m in metas:
    print(f"  {m.name:45s} | {m.kind.name:15s} | {m.frame_count:8d} frames")
    total += m.frame_count
print(f"\n  TOTAL FRAMES ACROSS ALL DATASETS: {total:,}")
if total > 20000:
    print("\n  [WARNING] Still uncapped. See section 3's LOADED FROM path above — "
          "compare it against the path checked in section 1. If they differ, THAT is the bug.")

print()
print("="*70)
print("5. WEIGHTS PRESENT?")
print("="*70)
for f in ['chakra_transformer_best.pth', 'combo1_best.pth']:
    p = f'{WORK}/weights/checkpoints/{f}'
    print(f"  {'OK' if os.path.exists(p) else 'MISSING'}: {p}")

1. FILE VERSION CHECK (raw file read from disk, exact path)
  [OK] batch ceiling fix (v4+): /kaggle/working/eval_harness/resource_monitor.py
       found: 'MAX_BATCH         = 8'
  [OK] compile-mode + checkpoint-remap fix (v4/v6+): /kaggle/working/eval_harness/eval_engine.py
       found: 'mode="default"'
       found: '_key_renames'
       found: 'ppd_pred.weight'
  [OK] recursive discovery + video cap/sequential-decode fix (v7): /kaggle/working/eval_harness/dataset_discovery.py
       found: '_discover_recursive'
       found: 'MAX_FRAMES_PER_VIDEO'
       found: 'IterableDataset'
       found: 'total_frames += min(n, MAX_FRAMES_PER_VIDEO)'

2. sys.path (import search order)
  [0] /kaggle/working/src
  [1] /kaggle/working/eval_harness
  [2] /kaggle/working
  [3] /kaggle/lib/kagglegym
  [4] /kaggle/lib
  [5] /usr/lib/python312.zip
  [6] /usr/lib/python3.12
  [7] /usr/lib/python3.12/lib-dynload
  [8] 
  [9] /usr/local/lib/python3.12/dist-packages
  [10] /usr/lib/python3/dist-packages
 